# SHAP Analysis — N-1 Security Classifier

XAI analysis for the SiKDD paper and the HCI/XAI evaluation with Jože and Ivana.
SHAP (SHapley Additive exPlanations) provides both global feature importance
and per-sample explanations for the Random Forest N-1 security classifier.

**Critical:** SHAP values are computed on the TEST set only to avoid data leakage.

Model: `RandomForestClassifier(n_estimators=400, random_state=42)`  
Split: temporal — train Jan–Sep 2023, test Oct–Dec 2023 (2208 test samples)  
Libraries: pandas, numpy, sklearn, matplotlib, shap

> Install: `pip install shap` if not already available.

## 1. Load data and train model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH  = Path('../data/simulation_security_labels_n-1.csv')
LABEL_COL  = 'status'
EXCLUDE_COLS = [
    'timestamp',
    'max_line_loading_percent_basecase',
    'min_bus_voltage_pu_basecase',
    'max_bus_voltage_pu_basecase',
    'max_line_loading_percent_contingency',
    'min_bus_voltage_pu_contingency',
    'max_bus_voltage_pu_contingency',
]

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS and c != LABEL_COL]

df_sorted  = df.sort_values('timestamp').reset_index(drop=True)
SPLIT_DATE = pd.Timestamp('2023-10-01 00:00:00')
df_train   = df_sorted[df_sorted['timestamp'] < SPLIT_DATE]
df_test    = df_sorted[df_sorted['timestamp'] >= SPLIT_DATE].reset_index(drop=True)

X_train    = df_train[feature_cols].values
y_train    = df_train[LABEL_COL].values
X_test     = df_test[feature_cols].values
y_test     = df_test[LABEL_COL].values
X_test_df  = df_test[feature_cols].copy()  # DataFrame for SHAP plots

print(f'n_features: {len(feature_cols)}')
print(f'Train:      {len(df_train)} rows ({str(df_train["timestamp"].min().date())} to {str(df_train["timestamp"].max().date())})')
print(f'Test:       {len(df_test)} rows ({str(df_test["timestamp"].min().date())} to {str(df_test["timestamp"].max().date())})')

clf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

insecure_idx = list(clf.classes_).index('insecure')
y_pred       = clf.predict(X_test)
y_prob       = clf.predict_proba(X_test)
y_prob_ins   = y_prob[:, insecure_idx]
y_bin        = (y_test == 'insecure').astype(int)

print(f'\nSanity check (temporal split):')
print(f'  accuracy:          {accuracy_score(y_test, y_pred):.4f}')
print(f'  recall_insecure:   {recall_score(y_test, y_pred, pos_label="insecure", zero_division=0):.4f}')
print(f'  roc_auc:           {roc_auc_score(y_bin, y_prob_ins):.4f}')

c:\Users\Gasper\Documents\Projekti\IJS\smart-energy-ea\venv-smart-energy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


n_features: 265
Train:      6561 rows (2023-01-01 to 2023-09-30)
Test:       2208 rows (2023-10-01 to 2023-12-31)

Sanity check (temporal split):
  accuracy:          0.9094
  recall_insecure:   0.8379
  roc_auc:           0.9783


## 2. Compute SHAP values

SHAP values are computed on the **test set only** (2208 samples, Oct–Dec 2023)
to prevent data leakage. This may take a few minutes for 400 trees × 265 features.

In [2]:
print('Computing SHAP values (TreeExplainer on test set)...')
explainer   = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Handle both SHAP API variants:
#   list of 2 arrays [insecure, secure] (older API)
#   3D array (n_samples, n_features, n_classes) (newer API)
if isinstance(shap_values, list):
    shap_vals_ins = shap_values[insecure_idx]          # shape: (n_test, n_features)
    base_value    = explainer.expected_value[insecure_idx]
else:
    shap_vals_ins = shap_values[:, :, insecure_idx]
    ev = explainer.expected_value
    base_value = ev[insecure_idx] if hasattr(ev, '__len__') else float(ev)

print(f'Done.')
print(f'shap_vals_ins shape:          {shap_vals_ins.shape}')
print(f'Base value (E[P(insecure)]):  {base_value:.4f}')

Computing SHAP values (TreeExplainer on test set)...
Done.
shap_vals_ins shape:          (2208, 265)
Base value (E[P(insecure)]):  0.4783


## 3. Global feature importance (SHAP)

Mean absolute SHAP value per feature, compared with Gini importance.

In [3]:
# SHAP importance
mean_abs_shap = np.abs(shap_vals_ins).mean(axis=0)
shap_imp_df = pd.DataFrame({
    'feature':      feature_cols,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_imp_df['rank'] = shap_imp_df.index + 1

# Gini importance (for comparison)
gini_df = pd.DataFrame({
    'feature':          feature_cols,
    'gini_importance':  clf.feature_importances_,
}).sort_values('gini_importance', ascending=False).reset_index(drop=True)
gini_df['gini_rank'] = gini_df.index + 1

# Top 20 SHAP table
print('Top 20 features by mean |SHAP| (insecure class):')
print(f'{"Rank":>4}  {"Feature":<45}  {"mean |SHAP|":>12}')
print('-' * 65)
for _, row in shap_imp_df.head(20).iterrows():
    print(f'{int(row["rank"]):>4}  {row["feature"]:<45}  {row["mean_abs_shap"]:>12.6f}')

# SHAP vs Gini rank comparison for top 10
print()
print('Top 10 comparison — SHAP rank vs Gini rank:')
print(f'{"Feature":<45}  {"SHAP rank":>10}  {"Gini rank":>10}')
print('-' * 68)
for _, row in shap_imp_df.head(10).iterrows():
    feat      = row['feature']
    shap_rank = int(row['rank'])
    gr        = gini_df[gini_df['feature'] == feat]['gini_rank']
    gini_rank = int(gr.values[0]) if len(gr) else -1
    print(f'{feat:<45}  {shap_rank:>10}  {gini_rank:>10}')

# Bar plot (top 30, coloured by group)
def feat_color(name):
    if name.startswith('load_'):  return '#4C72B0'
    if name.startswith('gen_'):   return '#DD8452'
    return '#55A868'

top30_shap = shap_imp_df.head(30).iloc[::-1]
colors     = [feat_color(f) for f in top30_shap['feature']]

plt.figure(figsize=(10, 9))
plt.barh(top30_shap['feature'], top30_shap['mean_abs_shap'], color=colors)
plt.xlabel('Mean |SHAP value| — impact on P(insecure)')
plt.title('Top 30 Features by Mean Absolute SHAP Value (Insecure Class)')
plt.legend(handles=[
    mpatches.Patch(color='#4C72B0', label='load_*'),
    mpatches.Patch(color='#DD8452', label='gen_*'),
    mpatches.Patch(color='#55A868', label='sgen_*'),
], loc='lower right')
plt.tight_layout()

out = FIGURES_DIR / 'shap_importance_bar.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'\nFigure saved to {out}')

Top 20 features by mean |SHAP| (insecure class):
Rank  Feature                                         mean |SHAP|
-----------------------------------------------------------------
   1  gen_48_p_mw                                        0.010385
   2  gen_90_p_mw                                        0.010339
   3  gen_5_p_mw                                         0.009747
   4  gen_10_p_mw                                        0.009188
   5  gen_126_p_mw                                       0.008934
   6  gen_24_p_mw                                        0.008727
   7  gen_71_p_mw                                        0.008513
   8  gen_114_p_mw                                       0.008476
   9  gen_4_p_mw                                         0.008157
  10  gen_23_p_mw                                        0.007996
  11  gen_113_p_mw                                       0.007923
  12  gen_96_p_mw                                        0.007863
  13  gen_108_p_mw         

## 4. SHAP beeswarm plot

Each dot is one test sample. X-axis is the SHAP value (positive = pushes toward insecure).
Color encodes feature value: red = high value, blue = low value.

In [4]:
shap.summary_plot(shap_vals_ins, X_test_df, max_display=30, show=False)
out = FIGURES_DIR / 'shap_beeswarm.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Beeswarm plot saved to {out}')
print()
print('Interpretation:')
print('  Features with positive SHAP values (right side) push predictions toward insecure.')
print('  High values (red) of top generator features increase insecure risk; low values (blue) reduce it.')
print('  The spread of dots shows how strongly each feature varies across test samples.')

Beeswarm plot saved to ..\figures\shap_beeswarm.png

Interpretation:
  Features with positive SHAP values (right side) push predictions toward insecure.
  High values (red) of top generator features increase insecure risk; low values (blue) reduce it.
  The spread of dots shows how strongly each feature varies across test samples.


## 5. Per-sample explanation (operator view)

Three representative test samples with individual SHAP explanations:
- **Sample a:** most confidently insecure (highest P(insecure))
- **Sample b:** most confidently secure (lowest P(insecure))
- **Sample c:** most uncertain (P(insecure) closest to 0.5)

In [5]:
idx_a = int(np.argmax(y_prob_ins))
idx_b = int(np.argmin(y_prob_ins))
idx_c = int(np.argmin(np.abs(y_prob_ins - 0.5)))


def print_sample(idx, label):
    sv       = shap_vals_ins[idx]
    ts       = df_test['timestamp'].iloc[idx]
    true_lbl = y_test[idx]
    pred_lbl = y_pred[idx]
    p        = y_prob_ins[idx]
    sorted_idx = np.argsort(sv)[::-1]

    print(f'--- {label} ---')
    print(f'  Timestamp:   {ts}')
    print(f'  True label:  {true_lbl}   Predicted: {pred_lbl}   P(insecure): {p:.4f}')
    print(f'  Top 5 pushing toward INSECURE (+ SHAP):')
    for i in sorted_idx[:5]:
        print(f'    {feature_cols[i]:<45}  {sv[i]:+.5f}')
    print(f'  Top 5 pushing toward SECURE (- SHAP):')
    for i in sorted_idx[-5:][::-1]:
        print(f'    {feature_cols[i]:<45}  {sv[i]:+.5f}')
    print()


def plot_sample_waterfall(idx, label, save_name):
    sv     = shap_vals_ins[idx]
    p      = y_prob_ins[idx]
    true_l = y_test[idx]

    top_idx = np.argsort(np.abs(sv))[::-1][:12]
    feats   = [feature_cols[i] for i in top_idx]
    vals    = sv[top_idx]
    colors  = ['#d62728' if v > 0 else '#1f77b4' for v in vals]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(list(reversed(feats)), list(reversed(vals)), color=list(reversed(colors)))
    ax.axvline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_xlabel('SHAP value (impact on P(insecure))')
    ax.set_title(f'{label}\ntrue={true_l},  P(insecure)={p:.3f}  (base={base_value:.3f})')
    ax.legend(handles=[
        mpatches.Patch(color='#d62728', label='toward insecure (+)'),
        mpatches.Patch(color='#1f77b4', label='toward secure (-)'),
    ], loc='lower right', fontsize=8)
    plt.tight_layout()

    out = FIGURES_DIR / save_name
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close('all')
    print(f'  Saved: {out}')


print_sample(idx_a, 'a) Most confidently INSECURE')
plot_sample_waterfall(idx_a, 'Sample a — Most confidently insecure', 'shap_sample_a.png')
print('  Operator interpretation: The features with the largest positive SHAP values are the')
print('  primary reason the model flags this operating point as insecure. The operator can')
print('  prioritise checking whether these generators are actually at the values shown.')
print()

print_sample(idx_b, 'b) Most confidently SECURE')
plot_sample_waterfall(idx_b, 'Sample b — Most confidently secure', 'shap_sample_b.png')
print('  Operator interpretation: All major features push strongly toward secure. The model')
print('  is very confident — this operating point is unlikely to require Digital Twin simulation.')
print()

print_sample(idx_c, 'c) Most UNCERTAIN (P(insecure) closest to 0.5)')
plot_sample_waterfall(idx_c, 'Sample c — Most uncertain (AL candidate)', 'shap_sample_c.png')
print('  Operator interpretation: Competing positive and negative contributions cancel out,')
print('  leaving the model uncertain. This is exactly the type of operating point that Active')
print('  Learning should prioritise for Digital Twin simulation to resolve the uncertainty.')

--- a) Most confidently INSECURE ---
  Timestamp:   2023-10-06 21:00:00
  True label:  insecure   Predicted: insecure   P(insecure): 1.0000
  Top 5 pushing toward INSECURE (+ SHAP):
    gen_48_p_mw                                    +0.01605
    gen_90_p_mw                                    +0.01365
    gen_5_p_mw                                     +0.01282
    gen_10_p_mw                                    +0.01250
    gen_126_p_mw                                   +0.01246
  Top 5 pushing toward SECURE (- SHAP):
    sgen_56_p_mw                                   -0.00334
    sgen_58_p_mw                                   -0.00329
    sgen_45_p_mw                                   -0.00306
    sgen_48_p_mw                                   -0.00302
    sgen_46_p_mw                                   -0.00294

  Saved: ..\figures\shap_sample_a.png
  Operator interpretation: The features with the largest positive SHAP values are the
  primary reason the model flags this operating point

## 6. Feature group SHAP analysis

In [6]:
def feat_group(name):
    if name.startswith('load_'):  return 'load_'
    if name.startswith('gen_'):   return 'gen_'
    return 'sgen_'

shap_imp_df['group'] = shap_imp_df['feature'].apply(feat_group)

group_shap = (
    shap_imp_df.groupby('group')
    .agg(
        n_features     =('mean_abs_shap', 'count'),
        total_shap     =('mean_abs_shap', 'sum'),
        mean_abs_shap  =('mean_abs_shap', 'mean'),
    )
    .sort_values('total_shap', ascending=False)
)
grand_shap = group_shap['total_shap'].sum()

print('Feature group SHAP summary (compare with Gini results in feature_importance_analysis.ipynb):')
print(f'{"Group":<8}  {"n_features":>10}  {"mean |SHAP|":>12}  {"total_shap":>12}  {"% total":>8}')
print('-' * 58)
for group, row in group_shap.iterrows():
    pct = row['total_shap'] / grand_shap * 100
    print(
        f'{group:<8}  {int(row["n_features"]):>10}  '
        f'{row["mean_abs_shap"]:>12.6f}  '
        f'{row["total_shap"]:>12.4f}  '
        f'{pct:>7.1f}%'
    )

dominant_shap = group_shap.index[0]
dominant_pct  = group_shap['total_shap'].iloc[0] / grand_shap * 100
print(f'\nDominant group by SHAP: {dominant_shap} ({dominant_pct:.1f}%)')

Feature group SHAP summary (compare with Gini results in feature_importance_analysis.ipynb):
Group     n_features   mean |SHAP|    total_shap   % total
----------------------------------------------------------
gen_             135      0.003015        0.4070     61.2%
sgen_            110      0.001992        0.2191     32.9%
load_             20      0.001958        0.0392      5.9%

Dominant group by SHAP: gen_ (61.2%)


## 7. Interpretation for paper

In [7]:
top_shap_feat = shap_imp_df.iloc[0]['feature']
top_gini_feat = gini_df.iloc[0]['feature']
top10_shap = shap_imp_df['feature'].head(10).tolist()
top10_gini = gini_df['feature'].head(10).tolist()
n_overlap   = len(set(top10_shap) & set(top10_gini))

print('=== Interpretation for paper ===')
print()
print(
    f'1. SHAP vs Gini agreement: {n_overlap}/10 top features overlap between SHAP and Gini rankings. '
    f'SHAP top feature: {top_shap_feat}; Gini top feature: {top_gini_feat}. '
    f'This cross-validation of two distinct importance methods strengthens confidence '
    f'that the identified features genuinely drive N-1 security predictions.'
)
print()
print(
    f'2. Dominant feature group: The {dominant_shap} group accounts for {dominant_pct:.1f}% '
    f'of total mean |SHAP|, consistent with Gini results. '
    f'Generator output values are the primary determinants of N-1 security state, '
    f'reflecting that generation levels drive line loadings and voltage profiles.'
)
print()
print(
    '3. Per-sample analysis for operators: The most confidently insecure sample shows '
    'specific generator features with large positive SHAP values — a direct actionable '
    'explanation ("high output on gen_X increases insecure risk"). The uncertain sample '
    'shows competing contributions that cancel out, identifying it as the highest-value '
    'Active Learning candidate for Digital Twin simulation.'
)
print()
print(
    '4. XAI / HCI evaluation support: SHAP enables both global explanations '
    '(which features matter overall?) and local explanations (why is THIS sample insecure?). '
    'For the Jože/Ivana HCI evaluation, per-sample SHAP bars can be shown alongside model '
    'predictions to test whether operators make better selection decisions with vs without XAI.'
)
print()
print(
    '5. Paper recommendation: Report the beeswarm plot (figures/shap_beeswarm.png) as the '
    'primary XAI figure — it shows importance, effect direction, and value distribution in '
    'one view. Include Sample c (uncertain case) as the operator-facing explanation example. '
    'Mention SHAP/Gini convergence as a robustness argument for the identified feature set.'
)
print()
print(
    '6. AL motivation from XAI: SHAP confirms that uncertain samples have competing '
    'feature contributions rather than missing information, supporting the hypothesis '
    'that targeted Active Learning (simulating boundary-region operating points) will '
    'efficiently resolve model uncertainty with minimal Digital Twin simulation calls.'
)

=== Interpretation for paper ===

1. SHAP vs Gini agreement: 9/10 top features overlap between SHAP and Gini rankings. SHAP top feature: gen_48_p_mw; Gini top feature: gen_48_p_mw. This cross-validation of two distinct importance methods strengthens confidence that the identified features genuinely drive N-1 security predictions.

2. Dominant feature group: The gen_ group accounts for 61.2% of total mean |SHAP|, consistent with Gini results. Generator output values are the primary determinants of N-1 security state, reflecting that generation levels drive line loadings and voltage profiles.

3. Per-sample analysis for operators: The most confidently insecure sample shows specific generator features with large positive SHAP values — a direct actionable explanation ("high output on gen_X increases insecure risk"). The uncertain sample shows competing contributions that cancel out, identifying it as the highest-value Active Learning candidate for Digital Twin simulation.

4. XAI / HCI eva